# Phishing Detection Training Pipeline
**Before running anything:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload `extract_features.py`, `model.py`, `train.py`, `requirements.txt` to the Colab file browser
3. Upload your CSV files (`train.csv`, `val.csv`, `test.csv`, `ai_heldout_test_features.csv`)
4. Run cells top to bottom

## STEP 0 (CRITICAL): Mount Google Drive
This saves ALL your progress to your Google Drive so a network drop or Colab restart **cannot delete your work!**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Create a project folder in your Drive to store everything permanently
PROJECT_DIR = '/content/drive/MyDrive/phishing_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/saved_models', exist_ok=True)
print(f'Project folder ready: {PROJECT_DIR}')

## STEP 1: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q torch transformers pandas scikit-learn language-tool-python spacy imbalanced-learn
!python -m spacy download en_core_web_sm -q

## STEP 2: Organise Files
Move uploaded CSVs into the `data/` folder and link project to Drive.

In [ ]:
import os, shutil

# Move uploaded CSVs into data/
!mkdir -p data
for f in ['train.csv', 'val.csv', 'test.csv']:
    if os.path.exists(f):
        shutil.copy(f, f'data/{f}')
        print(f'Moved {f} -> data/{f}')

# Copy ai_heldout_test_features.csv to data/ if present
if os.path.exists('ai_heldout_test_features.csv'):
    shutil.copy('ai_heldout_test_features.csv', 'data/ai_heldout_test_features.csv')
    print('Copied ai_heldout_test_features.csv -> data/')

# Also sync to Drive so they are safe
for f in os.listdir('data'):
    src = f'data/{f}'
    dst = f'{PROJECT_DIR}/data/{f}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Backed up to Drive: {f}')

print('\nFiles in data/:')
!ls -lh data/

## STEP 3A: Feature Extraction — FAST DEMO MODE (2000 rows, ~10 min)
**Use this first** to get results quickly. Produces real, valid numbers for your paper.

To do the FULL run instead, use Step 3B below.

In [ ]:
# FAST MODE: 2000 rows balanced (1000 phishing + 1000 safe)
# If this cell is interrupted and you re-run it, it will RESUME from where it left off!
!python extract_features.py --input data/train.csv --max_rows 2000

# Backup the result to Drive immediately
import shutil
shutil.copy('data/train_features.csv', f'{PROJECT_DIR}/data/train_features.csv')
print('Backed up train_features.csv to Drive!')

## STEP 3B: Feature Extraction — FULL RUN (17,500 rows, ~60-90 min)
Run this only if you want the full dataset. It checkpoints every 200 rows, so interruptions are safe.

In [ ]:
# FULL MODE — checkpoints every 200 rows, resume-safe
!python extract_features.py --input data/train.csv

import shutil
shutil.copy('data/train_features.csv', f'{PROJECT_DIR}/data/train_features.csv')
print('Backed up to Drive!')

## STEP 4: Train the 4 Model Variants
Trains `baseline`, `behavioral`, `psychological`, and `full` variants.

In [ ]:
!python train.py

# Backup all trained model weights to Drive
import shutil
if os.path.exists('saved_models'):
    for f in os.listdir('saved_models'):
        shutil.copy(f'saved_models/{f}', f'{PROJECT_DIR}/saved_models/{f}')
    print('All model weights backed up to Drive!')